In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib, os
os.makedirs('./models', exist_ok=True)

# 데이터 로드
train  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test   = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

# layout merge
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

train = train.merge(layout_clean, on='layout_id', how='left')
test  = test.merge(layout_clean, on='layout_id', how='left')

le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'].fillna('unknown'))
test['layout_type']  = le.transform(test['layout_type'].fillna('unknown'))

TARGET   = 'avg_delay_minutes_next_30m'
ID_COLS  = ['ID', 'layout_id', 'scenario_id']
feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
X = train[feature_cols]
y = train[TARGET]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_step1 = lgb.LGBMRegressor(
    n_estimators     = 200000,
    learning_rate    = 0.03,
    num_leaves       = 230,
    max_depth        = 12,
    min_child_samples= 37,
    subsample        = 0.6971592844853077,
    colsample_bytree = 0.6283895747824833,
    reg_alpha        = 0.3936177260538011,
    reg_lambda       = 1.0229688573580245,
    objective        = 'mae',
    metric           = 'mae',
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

model_step1.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(300), lgb.log_evaluation(5000)]
)

best_iter = model_step1.best_iteration_
print(f'Best iteration: {best_iter}')
print(f'Validation MAE: {np.mean(np.abs(y_val - model_step1.predict(X_val))):.4f}')

Training until validation scores don't improve for 300 rounds
[5000]	valid_0's l1: 6.60622
[10000]	valid_0's l1: 6.50177
[15000]	valid_0's l1: 6.47135
[20000]	valid_0's l1: 6.45084
[25000]	valid_0's l1: 6.44033
[30000]	valid_0's l1: 6.43002
[35000]	valid_0's l1: 6.41902
[40000]	valid_0's l1: 6.40988
[45000]	valid_0's l1: 6.39764
[50000]	valid_0's l1: 6.38789
[55000]	valid_0's l1: 6.38225
[60000]	valid_0's l1: 6.37705
[65000]	valid_0's l1: 6.37139
Early stopping, best iteration is:
[65673]	valid_0's l1: 6.37097
Best iteration: 65673
Validation MAE: 6.3710


In [2]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

kfold_params = {
    'n_estimators'     : 65673,
    'learning_rate'    : 0.03,
    'num_leaves'       : 230,
    'max_depth'        : 12,
    'min_child_samples': 37,
    'subsample'        : 0.6971592844853077,
    'colsample_bytree' : 0.6283895747824833,
    'reg_alpha'        : 0.3936177260538011,
    'reg_lambda'       : 1.0229688573580245,
    'objective'        : 'mae',
    'metric'           : 'mae',
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f'── Fold {fold+1} ──')
    X_tr,  y_tr  = X.iloc[tr_idx],  y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**kfold_params)
    model.fit(X_tr, y_tr)

    joblib.dump(model, f'C:/Users/user/dacon/Smart_Logistics/models/lgbm_optuna_lr03_fold{fold+1}.pkl')

    oof_preds[val_idx] = model.predict(X_val)
    test_preds        += model.predict(test[feature_cols]) / 5

    fold_mae = np.mean(np.abs(y_val - model.predict(X_val)))
    print(f'Fold {fold+1} MAE: {fold_mae:.4f}')

oof_mae = np.mean(np.abs(y - oof_preds))
print(f'\nOOF MAE: {oof_mae:.4f}')

── Fold 1 ──
Fold 1 MAE: 6.3710
── Fold 2 ──
Fold 2 MAE: 6.4964
── Fold 3 ──
Fold 3 MAE: 6.3869
── Fold 4 ──
Fold 4 MAE: 6.5246
── Fold 5 ──
Fold 5 MAE: 6.4592

OOF MAE: 6.4476


In [3]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': test_preds
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_v14_lgbm_optuna_lr03_kfold.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   16.735419
1  TEST_000001                   16.751457
2  TEST_000002                   19.576601
3  TEST_000003                   18.974186
4  TEST_000004                   18.425917
